# 🔢 Project 10.4 — Multi-Sensor Data Processor
**DSA for Mechatronics · Week 10 — Sorting Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from copy import deepcopy
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A data acquisition system collects readings from multiple sensor types.
We apply four non-comparison sorts and custom sorting patterns:

1. **Counting sort** — O(n + k) for integer sensor codes in a small range
2. **Radix sort (LSD)** — O(d·n) for multi-digit timestamps, digit by digit
3. **Relative sort** — order readings by a priority list, then sort remaining ascending
4. **Wiggle sort** — rearrange readings so highs and lows alternate
   (useful for oscillating control signals)


## Step 1 — Generate sensor datasets

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_CODES      = random.randint(30, 50)   # sensor event codes (small range)
CODE_RANGE   = random.randint(15, 30)   # 0..CODE_RANGE-1
N_TIMESTAMPS = random.randint(20, 35)   # multi-digit timestamps
N_READINGS   = random.randint(14, 20)   # for relative + wiggle sort
PRIORITY_LEN = random.randint(4, 7)

# 1) Sensor event codes (non-negative integers, small range)
codes = [random.randint(0, CODE_RANGE - 1) for _ in range(N_CODES)]

# 2) Multi-digit timestamps (4-digit ms values: 0–9999)
timestamps = [random.randint(100, 9999) for _ in range(N_TIMESTAMPS)]

# 3) Sensor reading values (floats 0.0–99.9 → use integers 0–99 for radix)
readings_int = [random.randint(0, 99) for _ in range(N_READINGS)]
readings_float = [round(v * 0.1 * random.uniform(0.5, 2.0), 2) for v in readings_int]

# 4) Priority list for relative sort
all_vals = sorted(set(readings_int))
priority_list = random.sample(all_vals, min(PRIORITY_LEN, len(all_vals)))

print(f"Dataset parameters:")
print(f"  Sensor codes     : {N_CODES} values  (range 0–{CODE_RANGE-1})")
print(f"  Timestamps       : {N_TIMESTAMPS} values  (4-digit ms)")
print(f"  Readings (int)   : {N_READINGS} values  (0–99)")
print(f"  Priority list    : {priority_list}")
print()
print(f"  Codes      : {codes}")
print(f"  Timestamps : {timestamps}")
print(f"  Readings   : {readings_int}")


## Step 2 — Counting sort (sensor event codes)

In [ ]:
def counting_sort(arr, k):
    """
    Counting sort for non-negative integers in range [0, k).

    Steps:
      1. count[v] = number of times value v appears.
      2. count[v] = number of elements ≤ v (prefix sums).
      3. Build output right-to-left for stability.

    O(n + k) time, O(n + k) space. NOT comparison-based.
    Ideal when k is small relative to n.
    """
    count = [0] * k
    for v in arr:
        count[v] += 1

    # Show frequency table
    freq_table = [(v, c) for v, c in enumerate(count) if c > 0]

    # Prefix sums (cumulative count)
    for i in range(1, k):
        count[i] += count[i - 1]

    # Build output (right-to-left for stability)
    output = [0] * len(arr)
    for v in reversed(arr):
        count[v] -= 1
        output[count[v]] = v

    return output, freq_table

sorted_codes, freq_table = counting_sort(codes, CODE_RANGE)

print(f"Counting sort — sensor event codes (n={N_CODES}, k={CODE_RANGE}):")
print()
print(f"  Frequency table:")
print(f"  {'Code':>6}  {'Count':>6}  Bar")
print(f"  {'─'*6}  {'─'*6}  {'─'*20}")
for v, c in freq_table:
    bar = "█" * c
    print(f"  {v:6d}  {c:6d}  {bar}")

print()
print(f"  Sorted codes (first 20): {sorted_codes[:20]}{'...' if N_CODES > 20 else ''}")
print(f"  Correct: {'✅ YES' if sorted_codes == sorted(codes) else '❌ NO'}")

most_common_code = max(freq_table, key=lambda x: x[1])
print(f"  Most frequent code  : {most_common_code[0]}  (appears {most_common_code[1]} times)")
unique_codes = len(freq_table)
print(f"  Unique codes        : {unique_codes} / {CODE_RANGE}")


## Step 3 — Radix sort LSD (timestamps)

In [ ]:
def radix_sort_lsd(arr, base=10):
    """
    Least-Significant-Digit (LSD) Radix Sort.

    Process digits from rightmost (ones) to leftmost.
    For each digit position, perform a STABLE counting sort.
    Stability is critical: elements equal in the current digit must
    keep the order established by previous (less-significant) digits.

    O(d * (n + base)) time where d = number of digits.
    """
    if not arr: return arr
    a = arr[:]
    max_val = max(a)
    d = len(str(max_val))   # number of digit positions

    digit_passes = []

    exp = 1
    for digit_pos in range(d):
        # Stable counting sort by current digit
        count = [0] * base
        for v in a:
            digit = (v // exp) % base
            count[digit] += 1

        # Snapshot for trace
        digit_snapshot = [(v, (v // exp) % base) for v in a[:8]]
        digit_passes.append((digit_pos, digit_snapshot, a[:]))

        for i in range(1, base):
            count[i] += count[i - 1]
        output = [0] * len(a)
        for v in reversed(a):
            digit = (v // exp) % base
            count[digit] -= 1
            output[count[digit]] = v
        a = output
        exp *= base

    return a, digit_passes

sorted_ts, ts_passes = radix_sort_lsd(timestamps)

print(f"Radix sort LSD — timestamps (n={N_TIMESTAMPS}):")
print(f"  Max value   : {max(timestamps)}  → {len(str(max(timestamps)))} digit passes")
print()
for pass_i, (dpos, snap, state_before) in enumerate(ts_passes):
    place = ["ones","tens","hundreds","thousands"][dpos] if dpos < 4 else f"10^{dpos}"
    print(f"  Pass {pass_i+1} (sort by {place} digit):")
    print(f"    Before (first 8): {state_before[:8]}")
    print(f"    Digit extracted  : {[d for _, d in snap]}")

print()
print(f"  After all passes  : {sorted_ts}")
print(f"  Correct           : {'✅ YES' if sorted_ts == sorted(timestamps) else '❌ NO'}")


## Step 4 — Relative sort + wiggle sort

In [ ]:
def relative_sort(arr, priority):
    """
    Sort arr such that elements in priority appear first (in priority's order),
    then remaining elements in ascending order.
    O(n log n) using a custom key.
    """
    prio_map = {v: i for i, v in enumerate(priority)}
    def sort_key(x):
        if x in prio_map:
            return (0, prio_map[x], x)   # group 0: priority order
        return (1, 0, x)                  # group 1: ascending
    return sorted(arr, key=sort_key)

def wiggle_sort(arr):
    """
    Rearrange so arr[0] < arr[1] > arr[2] < arr[3] > ...
    (valley at even indices, peak at odd indices)

    One-pass approach:
      For each adjacent pair (i, i+1):
        - If i is even (should be valley) and arr[i] > arr[i+1]: swap
        - If i is odd  (should be peak)  and arr[i] < arr[i+1]: swap
    O(n), in-place.
    """
    a = arr[:]
    swaps_w = 0
    for i in range(len(a) - 1):
        if (i % 2 == 0 and a[i] > a[i+1]) or (i % 2 == 1 and a[i] < a[i+1]):
            a[i], a[i+1] = a[i+1], a[i]
            swaps_w += 1
    return a, swaps_w

# Relative sort
rel_sorted = relative_sort(readings_int, priority_list)

print(f"Relative sort (n={N_READINGS}):")
print(f"  Priority list  : {priority_list}")
print(f"  Input readings : {readings_int}")
print(f"  Relative sorted: {rel_sorted}")
# Verify: priority elements come first in priority order
prio_set = set(priority_list)
prio_part = [v for v in rel_sorted if v in prio_set]
rest_part = [v for v in rel_sorted if v not in prio_set]
prio_order_ok = [priority_list.index(v) for v in prio_part] == sorted([priority_list.index(v) for v in prio_part])
rest_sorted_ok = rest_part == sorted(rest_part)
print(f"  Priority order correct : {'✅' if prio_order_ok else '❌'}")
print(f"  Remainder sorted asc   : {'✅' if rest_sorted_ok else '❌'}")

# Wiggle sort
wiggle, w_swaps = wiggle_sort(readings_int)

print()
print(f"Wiggle sort (n={N_READINGS}, swaps={w_swaps}):")
print(f"  Input    : {readings_int}")
print(f"  Wiggle   : {wiggle}")
# Verify wiggle property
valid_wiggle = all(
    (wiggle[i] < wiggle[i+1]) if i % 2 == 0 else (wiggle[i] > wiggle[i+1])
    for i in range(len(wiggle) - 1)
)
print(f"  Valid wiggle pattern : {'✅ YES' if valid_wiggle else '❌ NO'}")
# Visual pattern
pattern = ""
for i, v in enumerate(wiggle):
    if i == 0: pattern += str(v)
    else: pattern += (" < " if i%2==0 else " > ") + str(v)
print(f"  Pattern  : {pattern[:80]}{'...' if len(pattern)>80 else ''}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " MULTI-SENSOR DATA PROCESSOR — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  COUNTING SORT" + " "*(W-15) + "║")
print(f"║  {'n (codes)':<28}: {N_CODES:<26}║")
print(f"║  {'Value range k':<28}: {CODE_RANGE:<26}║")
print(f"║  {'Unique codes':<28}: {unique_codes:<26}║")
print(f"║  {'Most frequent code':<28}: {most_common_code[0]} (×{most_common_code[1]}){'':<18}║")
correct_cs = sorted_codes == sorted(codes)
print(f"║  {'Result correct':<28}: {'YES ✅' if correct_cs else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  RADIX SORT LSD" + " "*(W-16) + "║")
print(f"║  {'n (timestamps)':<28}: {N_TIMESTAMPS:<26}║")
print(f"║  {'Digit passes':<28}: {len(ts_passes):<26}║")
correct_rs = sorted_ts == sorted(timestamps)
print(f"║  {'Result correct':<28}: {'YES ✅' if correct_rs else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  RELATIVE SORT" + " "*(W-15) + "║")
print(f"║  {'Priority list length':<28}: {PRIORITY_LEN:<26}║")
print(f"║  {'Priority order correct':<28}: {'YES ✅' if prio_order_ok else 'NO ❌':<26}║")
print(f"║  {'Remainder sorted asc':<28}: {'YES ✅' if rest_sorted_ok else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  WIGGLE SORT" + " "*(W-13) + "║")
print(f"║  {'n (readings)':<28}: {N_READINGS:<26}║")
print(f"║  {'Swaps used':<28}: {w_swaps:<26}║")
print(f"║  {'Valid wiggle pattern':<28}: {'YES ✅' if valid_wiggle else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about sorting in this context?

*Your answer here:*

---

**Q2 — Which sorting concept did you find most important, and why?**
Pick one algorithm or pattern (merge sort, quickselect, interval merging, counting sort…) and explain what problem it solves best.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
